# Nightly Incremental Ingest

Keeps the bronze tables current. This is the scheduled counterpart to `01_data_ingestion.ipynb`,
which stays the one-time historical backfill.

Differences that matter:

- **Rolling window, not a fixed range.** Re-requests the last `lookback_days`, because GHCND
  publishes on a ~5 day lag and revises already-published days after the fact.
- **`MERGE`, not `DROP` + `overwrite`.** Idempotent — running it twice in a night converges to
  the same table instead of duplicating rows.
- **Pinned stations.** The list is explicit and lives in `noaa_client.DEFAULT_STATIONS` rather
  than being picked dynamically, so a shift in NOAA's station metadata can't splice another
  location's readings into the series. Bronze is *long* in the station dimension — more stations
  means more rows, not more columns, so the `(station, date)` merge key is unchanged.
- **NWS accumulates.** Appends to `nws_forecast_history` with an `ingested_at` column instead of
  overwriting a snapshot, which is what stage-4 drift monitoring needs.

Run `01_data_ingestion.ipynb` once before scheduling this. Design notes are in this folder's `README.md`.

## 0. Parameters

All run-time configuration is exposed as widgets so this can be driven by a Databricks Job task's parameters.

In [ ]:
dbutils.widgets.text('catalog', 'mlo', 'Unity Catalog catalog')
dbutils.widgets.text('schema', 'weather_mlops', 'Bronze schema')
dbutils.widgets.text('station_ids', '', 'Station ids, comma separated (blank = module default)')
dbutils.widgets.text('datatypes', '', 'NOAA datatypes, comma separated (blank = module default)')
dbutils.widgets.text('lookback_days', '10', 'Days to re-request each run')
dbutils.widgets.text('lat', '41.85', 'NWS latitude')
dbutils.widgets.text('lon', '-87.65', 'NWS longitude')
dbutils.widgets.text('nws_contact_email', 'klkendall@uchicago.edu', 'Contact email for NWS User-Agent')
dbutils.widgets.text('secret_scope', 'mlo', 'Secret scope holding the NOAA token')
dbutils.widgets.text('secret_key', 'WEATHER_API_KEY', 'Secret key holding the NOAA token')

In [ ]:
import os
import sys
from datetime import date, timedelta

# Files in Repos puts this notebook's own directory on sys.path. Add the repo root so
# `data_pipelines` is importable by package name — the same import statement then works
# from 01_data_ingestion.ipynb, which sits at the repo root.
REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from data_pipelines.noaa_client import (
    DEFAULT_DATATYPES,
    DEFAULT_STATIONS,
    get_ghcnd_daily_multi,
    make_headers,
)


def _csv_widget(name, fallback):
    raw = dbutils.widgets.get(name).strip()
    return tuple(v.strip() for v in raw.split(',') if v.strip()) if raw else tuple(fallback)


CATALOG = dbutils.widgets.get('catalog')
SCHEMA = dbutils.widgets.get('schema')
LOOKBACK_DAYS = int(dbutils.widgets.get('lookback_days'))

# Blank widgets fall back to the module, so the frozen contract lives in version control
# rather than in job parameters that can drift without anyone reviewing the change.
STATION_IDS = _csv_widget('station_ids', DEFAULT_STATIONS)
DATATYPES = _csv_widget('datatypes', DEFAULT_DATATYPES)

NOAA_TABLE = f'{CATALOG}.{SCHEMA}.noaa_historical_daily'
NWS_TABLE = f'{CATALOG}.{SCHEMA}.nws_forecast_history'

HEADERS = make_headers(
    dbutils.secrets.get(scope=dbutils.widgets.get('secret_scope'),
                        key=dbutils.widgets.get('secret_key'))
)

print(f'{len(STATION_IDS)} stations, {len(DATATYPES)} datatypes')
print('datatypes:', ', '.join(DATATYPES))
print('target   :', NOAA_TABLE)

## 1. Preconditions

This job only does incremental updates — it never creates the bronze table. If the backfill hasn't
run, fail immediately with a useful message rather than half-building a table on a schedule.

In [ ]:
if not spark.catalog.tableExists(NOAA_TABLE):
    raise RuntimeError(
        f'{NOAA_TABLE} does not exist. This job only performs incremental updates — '
        'run 01_data_ingestion.ipynb once to backfill the history first.'
    )

existing = spark.table(NOAA_TABLE)
missing = [c for c in ('station', 'date', *DATATYPES) if c not in existing.columns]
if missing:
    raise RuntimeError(
        f'{NOAA_TABLE} is missing {missing}, which the datatypes widget expects. '
        'The bronze schema and the datatypes parameter have diverged — re-run the backfill '
        'with the new column set before scheduling.'
    )

print(f'{NOAA_TABLE} present with columns: {existing.columns}')

## 2. NOAA — rolling window pull

The window is deliberately wider than one day. GHCND finalises data on a lag, so a strict
"yesterday" request usually returns nothing, and NOAA revises days it has already published.
Re-requesting the last `lookback_days` covers both, and `MERGE` makes the overlap harmless.

In [ ]:
END_DATE = date.today()
START_DATE = END_DATE - timedelta(days=LOOKBACK_DAYS)

print(f'window {START_DATE}..{END_DATE}\n')

updates, failures = get_ghcnd_daily_multi(
    STATION_IDS,
    START_DATE.isoformat(),
    END_DATE.isoformat(),
    headers=HEADERS,
    datatypes=DATATYPES,
    skip_errors=True,  # one station down overnight shouldn't cost the other nine
    progress=lambda station_id, n: print(f'  {station_id:<20} {n:>4} days'),
)

for station_id, message in failures:
    print(f'  {station_id:<20} FAILED  {message}')

# skip_errors would otherwise turn a total outage into a quiet no-op run.
if failures and len(failures) == len(STATION_IDS):
    raise RuntimeError('every station failed — this is an outage, not an empty night')

print(f'\n{len(updates)} station-days to merge')
updates.head()

In [ ]:
if updates.empty:
    # Normal, not an error: GHCND publishes on a lag, and a short window can legitimately
    # be empty (holidays, station outages). Leave the table untouched.
    print('No NOAA rows in window — nothing to merge.')
else:
    spark.createDataFrame(updates).createOrReplaceTempView('noaa_updates')
    # UPDATE SET * means NOAA wins on conflict, which is how revisions land. It also means a
    # value NOAA now reports as null overwrites what we had. NOAA is the system of record for
    # this window, so that's intended — see README.md.
    spark.sql(f'''
        MERGE INTO {NOAA_TABLE} AS t
        USING noaa_updates AS s
           ON t.station = s.station AND t.date = s.date
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    ''')
    print(f'merged {len(updates)} station-days into {NOAA_TABLE}')

## 3. NWS — append tonight's forecast

Each run appends a fresh forecast snapshot stamped with `ingested_at`. Over time this becomes the
forecast-vs-actual series that drift monitoring reads. Note this is a *different* table from the
`nws_forecast_snapshot` that `01` overwrites — that notebook's behaviour is unchanged.

In [ ]:
import pandas as pd
import requests
from pyspark.sql.functions import current_timestamp

NWS_HEADERS = {
    'User-Agent': f"(mlops-weather-project, contact: {dbutils.widgets.get('nws_contact_email')})",
    'Accept': 'application/geo+json',
}
LAT, LON = float(dbutils.widgets.get('lat')), float(dbutils.widgets.get('lon'))

point = requests.get(f'https://api.weather.gov/points/{LAT},{LON}', headers=NWS_HEADERS)
point.raise_for_status()
forecast_url = point.json()['properties']['forecast']

resp = requests.get(forecast_url, headers=NWS_HEADERS)
resp.raise_for_status()
forecast = pd.json_normalize(resp.json()['properties']['periods'])

# json_normalize produces dotted names (probabilityOfPrecipitation.value). Delta tolerates them,
# but every downstream query would need backticks, so flatten to underscores here.
forecast.columns = [c.replace('.', '_') for c in forecast.columns]

(spark.createDataFrame(forecast)
      .withColumn('ingested_at', current_timestamp())
      .write.mode('append').option('mergeSchema', 'true')
      .saveAsTable(NWS_TABLE))

print(f'appended {len(forecast)} forecast periods to {NWS_TABLE}')

## 4. Post-run checks

Cheap assertions that turn a silent data problem into a failed job run.

In [ ]:
# The MERGE key has to actually be unique, or every downstream join silently fans out.
dupes = spark.sql(f'''
    SELECT station, date, COUNT(*) AS n
    FROM {NOAA_TABLE}
    GROUP BY station, date
    HAVING COUNT(*) > 1
''').count()
assert dupes == 0, f'{dupes} duplicate (station, date) keys in {NOAA_TABLE}'

# Bronze should hold exactly the pinned stations: nothing extra spliced in by a station-id
# typo, and nothing silently missing because one has been failing for days.
present = {r[0] for r in spark.sql(f'SELECT DISTINCT station FROM {NOAA_TABLE}').collect()}
unexpected = present - set(STATION_IDS)
assert not unexpected, f'unpinned stations found in bronze: {sorted(unexpected)}'

absent = set(STATION_IDS) - present
if absent:
    print(f'WARNING: pinned stations with no rows at all: {sorted(absent)}')

# Per-station staleness. One station quietly falling behind is the failure mode a single
# table-wide max(date) would hide.
display(spark.sql(f'''
    SELECT station,
           COUNT(*) AS rows,
           MIN(date) AS min_date,
           MAX(date) AS max_date,
           DATEDIFF(CURRENT_DATE(), MAX(date)) AS days_behind
    FROM {NOAA_TABLE}
    GROUP BY station
    ORDER BY days_behind DESC, station
'''))